# Eval 3.2 - Relative Improvement Analysis

This notebook sits on top of `eval-3` and `eval-3.1`.

It does not introduce new models or a new split. Instead, it converts the existing metric tables
into percentage-gain comparisons for the key method families:

- flagship hybrid vs text-only
- flagship hybrid vs graph-only
- pooling vs no pooling
- corrected vs uncorrected


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "WikiCS" / "custom-wiki").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CUSTOM_WIKI_DIR = REPO_ROOT / "WikiCS" / "custom-wiki"
EVAL3_DIR = CUSTOM_WIKI_DIR / "cache" / "eval-3"
EVAL31_DIR = CUSTOM_WIKI_DIR / "cache" / "eval-3.1"
CACHE_DIR = CUSTOM_WIKI_DIR / "cache" / "eval-3.2"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Eval-3 dir:", EVAL3_DIR)
print("Eval-3.1 dir:", EVAL31_DIR)
print("Eval-3.2 dir:", CACHE_DIR)


Repo root: C:\programming\github-repos\graph-ending
Eval-3 dir: C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3
Eval-3.1 dir: C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.1
Eval-3.2 dir: C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.2


## Step 1 - Load and merge the revised metric tables


In [2]:
df3 = pd.read_csv(EVAL3_DIR / "combined_metrics.csv").copy()
df31 = pd.read_csv(EVAL31_DIR / "eval31_extension_metrics.csv").copy()

df3 = df3.rename(
    columns={
        "NDCG": "NDCG@100",
        "Recall": "Recall@100",
        "MAP": "MAP@100",
    }
)

df31 = df31.drop(columns=["Hit@10", "n_eval_pairs"])
df = df3.merge(df31, on="Model", how="inner")


def clean_model_name(name: str) -> str:
    return re.sub(r"^\d+\.\s*", "", str(name)).strip()


df["Model Clean"] = df["Model"].map(clean_model_name)
display(df)


,Model,Hit@1,Hit@10,Hit@50,Hit@100,MRR,NDCG@100,Recall@100,MAP@100,n_eval_pairs,...,n_recommendation_lists,top_k,NDCG@10,Recall@10,MAP@10,ROC-AUC,PR-AUC,n_positive_edges,n_negative_edges,Model Clean
0,10. BGE pooled + Node2Vec (corrected),0.010284,0.086736,0.314566,0.478090,0.040967,0.116046,0.478090,0.039236,86446,...,9022,20,0.040913,0.086736,0.027275,0.956345,0.955885,43223,43223,BGE pooled + Node2Vec (corrected)
1,5. BGE-M3 + Node2Vec (corrected),0.010712,0.086690,0.324064,0.498126,0.042058,0.120153,0.498126,0.040279,86446,...,9022,20,0.041304,0.086690,0.027778,0.958185,0.957947,43223,43223,BGE-M3 + Node2Vec (corrected)
2,9. BGE pooled + Node2Vec (uncorrected),0.007565,0.078349,0.326412,0.507415,0.037610,0.117589,0.507415,0.035782,86446,...,9022,20,0.035593,0.078349,0.022913,0.970989,0.971573,43223,43223,BGE pooled + Node2Vec (uncorrected)
3,4. BGE-M3 + Node2Vec (uncorrected),0.007577,0.078338,0.326400,0.507392,0.037634,0.117605,0.507392,0.035805,86446,...,9022,20,0.035608,0.078338,0.022934,0.971005,0.971594,43223,43223,BGE-M3 + Node2Vec (uncorrected)
4,2. Node2Vec (uncorrected),0.007542,0.078118,0.325244,0.505981,0.037487,0.117224,0.505981,0.035655,86446,...,9022,20,0.035508,0.078130,0.022870,0.971666,0.971390,43223,43223,Node2Vec (uncorrected)
5,3. Node2Vec (corrected),0.007519,0.078083,0.325244,0.505981,0.037469,0.117209,0.505981,0.035637,86446,...,9022,20,0.035464,0.078095,0.022823,0.971666,0.971376,43223,43223,Node2Vec (corrected)
6,7. GCN + BGE-M3 (rank fusion),0.008479,0.076325,0.231381,0.337737,0.033468,0.086187,0.337737,0.031694,86446,...,9022,20,0.035544,0.076325,0.023416,0.935649,0.936250,43223,43223,GCN + BGE-M3 (rank fusion)
7,1. BGE-M3 Only,0.013534,0.075689,0.196921,0.277260,0.036616,0.078753,0.277260,0.035146,86446,...,9022,20,0.040601,0.076383,0.029794,0.885149,0.888556,43223,43223,BGE-M3 Only
8,6. GCN Only,0.003841,0.034588,0.141186,0.229103,0.018029,0.052966,0.229103,0.016304,86446,...,9022,20,0.016149,0.034588,0.010660,0.895377,0.894558,43223,43223,GCN Only
9,8. GraphSAGE Only,0.001747,0.012227,0.049603,0.087153,0.007569,0.019899,0.087153,0.006138,86446,...,9022,20,0.005961,0.012227,0.004096,0.869078,0.856113,43223,43223,GraphSAGE Only


## Step 2 - Define metric directions and comparison pairs

We treat lower `Popularity Lift` and lower `Gini Index` as improvements. The rest of the selected
metrics are higher-is-better.


In [3]:
metric_directions = {
    "Hit@1": "higher",
    "Hit@10": "higher",
    "Hit@50": "higher",
    "Hit@100": "higher",
    "MRR": "higher",
    "NDCG@100": "higher",
    "Recall@100": "higher",
    "MAP@100": "higher",
    "NDCG@10": "higher",
    "Recall@10": "higher",
    "MAP@10": "higher",
    "ROC-AUC": "higher",
    "PR-AUC": "higher",
    "Popularity Lift": "lower",
    "Gini Index": "lower",
    "Aggregate Diversity": "higher",
    "Novelty": "higher",
    "Intra-List Distance": "higher",
}

focused_metrics = [
    "Hit@10",
    "NDCG@10",
    "Recall@10",
    "MAP@10",
    "ROC-AUC",
    "PR-AUC",
    "Popularity Lift",
    "Gini Index",
    "Aggregate Diversity",
    "Novelty",
    "Intra-List Distance",
]

comparisons = [
    {
        "Comparison": "Flagship hybrid vs text-only",
        "Candidate": "10. BGE pooled + Node2Vec (corrected)",
        "Baseline": "1. BGE-M3 Only",
    },
    {
        "Comparison": "Flagship hybrid vs graph-only",
        "Candidate": "10. BGE pooled + Node2Vec (corrected)",
        "Baseline": "2. Node2Vec (uncorrected)",
    },
    {
        "Comparison": "Pooling effect (corrected)",
        "Candidate": "10. BGE pooled + Node2Vec (corrected)",
        "Baseline": "5. BGE-M3 + Node2Vec (corrected)",
    },
    {
        "Comparison": "Pooling effect (uncorrected)",
        "Candidate": "9. BGE pooled + Node2Vec (uncorrected)",
        "Baseline": "4. BGE-M3 + Node2Vec (uncorrected)",
    },
    {
        "Comparison": "Correction effect (Node2Vec)",
        "Candidate": "3. Node2Vec (corrected)",
        "Baseline": "2. Node2Vec (uncorrected)",
    },
    {
        "Comparison": "Correction effect (hybrid)",
        "Candidate": "5. BGE-M3 + Node2Vec (corrected)",
        "Baseline": "4. BGE-M3 + Node2Vec (uncorrected)",
    },
    {
        "Comparison": "Correction effect (pooled hybrid)",
        "Candidate": "10. BGE pooled + Node2Vec (corrected)",
        "Baseline": "9. BGE pooled + Node2Vec (uncorrected)",
    },
]

model_rows = {row["Model"]: row for _, row in df.iterrows()}


## Step 3 - Compute relative improvements

For higher-is-better metrics:

`100 * (candidate - baseline) / baseline`

For lower-is-better metrics:

`100 * (baseline - candidate) / baseline`


In [4]:
def relative_improvement(candidate_value: float, baseline_value: float, direction: str) -> float:
    if pd.isna(candidate_value) or pd.isna(baseline_value):
        return np.nan
    if baseline_value == 0:
        return np.nan
    if direction == "higher":
        return 100.0 * (candidate_value - baseline_value) / baseline_value
    if direction == "lower":
        return 100.0 * (baseline_value - candidate_value) / baseline_value
    raise ValueError(f"Unknown direction: {direction}")


long_rows = []
wide_rows = []

for comparison in comparisons:
    candidate = model_rows[comparison["Candidate"]]
    baseline = model_rows[comparison["Baseline"]]

    wide_row = {
        "Comparison": comparison["Comparison"],
        "Candidate": candidate["Model Clean"],
        "Baseline": baseline["Model Clean"],
    }

    for metric, direction in metric_directions.items():
        cand_val = float(candidate[metric])
        base_val = float(baseline[metric])
        gain = relative_improvement(cand_val, base_val, direction)

        long_rows.append(
            {
                "Comparison": comparison["Comparison"],
                "Candidate": candidate["Model Clean"],
                "Baseline": baseline["Model Clean"],
                "Metric": metric,
                "Direction": direction,
                "Baseline Value": base_val,
                "Candidate Value": cand_val,
                "Relative Improvement (%)": gain,
            }
        )

        if metric in focused_metrics:
            wide_row[metric] = gain

    wide_rows.append(wide_row)

df_long = pd.DataFrame(long_rows)
df_wide = pd.DataFrame(wide_rows)

display(df_wide)


,Comparison,Candidate,Baseline,Hit@10,NDCG@10,Recall@10,MAP@10,ROC-AUC,PR-AUC,Popularity Lift,Gini Index,Aggregate Diversity,Novelty,Intra-List Distance
0,Flagship hybrid vs text-only,BGE pooled + Node2Vec (corrected),BGE-M3 Only,14.595751,0.769261,13.554445,-8.455192,8.043326e+00,7.577250,31.482347,20.532943,3.860614,3.090124,17.750588
1,Flagship hybrid vs graph-only,BGE pooled + Node2Vec (corrected),Node2Vec (uncorrected),11.032134,15.222400,11.015694,19.259508,-1.576783e+00,-1.596212,0.542559,8.293275,3.282874,1.617259,-4.303720
2,Pooling effect (corrected),BGE pooled + Node2Vec (corrected),BGE-M3 + Node2Vec (corrected),0.053376,-0.947115,0.053376,-1.811709,-1.920063e-01,-0.215298,-1.682066,0.412180,0.514732,-0.031404,1.309042
3,Pooling effect (uncorrected),BGE pooled + Node2Vec (uncorrected),BGE-M3 + Node2Vec (uncorrected),0.014767,-0.042630,0.014767,-0.092373,-1.670701e-03,-0.002072,0.022784,-0.170501,0.197062,0.007872,0.027811
4,Correction effect (Node2Vec),Node2Vec (corrected),Node2Vec (uncorrected),-0.044425,-0.123427,-0.044418,-0.207102,-8.263130e-08,-0.001430,0.000000,0.000000,0.000000,0.000000,0.000000
5,Correction effect (hybrid),BGE-M3 + Node2Vec (corrected),BGE-M3 + Node2Vec (uncorrected),10.661548,15.997512,10.661548,21.120334,-1.320354e+00,-1.404549,2.379048,10.379669,0.931566,1.740745,-5.296000
6,Correction effect (pooled hybrid),BGE pooled + Node2Vec (corrected),BGE pooled + Node2Vec (uncorrected),10.704267,14.947885,10.704267,19.035943,-1.508180e+00,-1.614785,0.714379,10.900981,1.251564,1.700789,-4.082961


## Step 4 - Save eval-3.2 artifacts


In [5]:
df.to_csv(CACHE_DIR / "merged_metrics.csv", index=False)
df_long.to_csv(CACHE_DIR / "relative_improvements_long.csv", index=False)
df_wide.to_csv(CACHE_DIR / "relative_improvements_wide.csv", index=False)

summary = {
    "n_models": int(len(df)),
    "n_comparisons": int(len(comparisons)),
    "focused_metrics": focused_metrics,
    "all_metrics": list(metric_directions.keys()),
}
with open(CACHE_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print("-", CACHE_DIR / "merged_metrics.csv")
print("-", CACHE_DIR / "relative_improvements_long.csv")
print("-", CACHE_DIR / "relative_improvements_wide.csv")
print("-", CACHE_DIR / "run_metadata.json")


Saved:
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.2\merged_metrics.csv
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.2\relative_improvements_long.csv
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.2\relative_improvements_wide.csv
- C:\programming\github-repos\graph-ending\WikiCS\custom-wiki\cache\eval-3.2\run_metadata.json


## Step 5 - Preview the long-form comparison table


In [6]:
print("Wide summary table:")
display(df_wide)

print("Long relative-improvement table:")
display(df_long)


Wide summary table:


,Comparison,Candidate,Baseline,Hit@10,NDCG@10,Recall@10,MAP@10,ROC-AUC,PR-AUC,Popularity Lift,Gini Index,Aggregate Diversity,Novelty,Intra-List Distance
0,Flagship hybrid vs text-only,BGE pooled + Node2Vec (corrected),BGE-M3 Only,14.595751,0.769261,13.554445,-8.455192,8.043326e+00,7.577250,31.482347,20.532943,3.860614,3.090124,17.750588
1,Flagship hybrid vs graph-only,BGE pooled + Node2Vec (corrected),Node2Vec (uncorrected),11.032134,15.222400,11.015694,19.259508,-1.576783e+00,-1.596212,0.542559,8.293275,3.282874,1.617259,-4.303720
2,Pooling effect (corrected),BGE pooled + Node2Vec (corrected),BGE-M3 + Node2Vec (corrected),0.053376,-0.947115,0.053376,-1.811709,-1.920063e-01,-0.215298,-1.682066,0.412180,0.514732,-0.031404,1.309042
3,Pooling effect (uncorrected),BGE pooled + Node2Vec (uncorrected),BGE-M3 + Node2Vec (uncorrected),0.014767,-0.042630,0.014767,-0.092373,-1.670701e-03,-0.002072,0.022784,-0.170501,0.197062,0.007872,0.027811
4,Correction effect (Node2Vec),Node2Vec (corrected),Node2Vec (uncorrected),-0.044425,-0.123427,-0.044418,-0.207102,-8.263130e-08,-0.001430,0.000000,0.000000,0.000000,0.000000,0.000000
5,Correction effect (hybrid),BGE-M3 + Node2Vec (corrected),BGE-M3 + Node2Vec (uncorrected),10.661548,15.997512,10.661548,21.120334,-1.320354e+00,-1.404549,2.379048,10.379669,0.931566,1.740745,-5.296000
6,Correction effect (pooled hybrid),BGE pooled + Node2Vec (corrected),BGE pooled + Node2Vec (uncorrected),10.704267,14.947885,10.704267,19.035943,-1.508180e+00,-1.614785,0.714379,10.900981,1.251564,1.700789,-4.082961


Long relative-improvement table:


,Comparison,Candidate,Baseline,Metric,Direction,Baseline Value,Candidate Value,Relative Improvement (%)
0,Flagship hybrid vs text-only,BGE pooled + Node2Vec (corrected),BGE-M3 Only,Hit@1,higher,0.013534,0.010284,-24.017094
1,Flagship hybrid vs text-only,BGE pooled + Node2Vec (corrected),BGE-M3 Only,Hit@10,higher,0.075689,0.086736,14.595751
2,Flagship hybrid vs text-only,BGE pooled + Node2Vec (corrected),BGE-M3 Only,Hit@50,higher,0.196921,0.314566,59.742701
3,Flagship hybrid vs text-only,BGE pooled + Node2Vec (corrected),BGE-M3 Only,Hit@100,higher,0.277260,0.478090,72.434079
4,Flagship hybrid vs text-only,BGE pooled + Node2Vec (corrected),BGE-M3 Only,MRR,higher,0.036616,0.040967,11.882505
...,...,...,...,...,...,...,...,...
121,Correction effect (pooled hybrid),BGE pooled + Node2Vec (corrected),BGE pooled + Node2Vec (uncorrected),Popularity Lift,lower,1.339688,1.330117,0.714379
122,Correction effect (pooled hybrid),BGE pooled + Node2Vec (corrected),BGE pooled + Node2Vec (uncorrected),Gini Index,lower,0.461295,0.411009,10.900981
123,Correction effect (pooled hybrid),BGE pooled + Node2Vec (corrected),BGE pooled + Node2Vec (uncorrected),Aggregate Diversity,higher,11186.000000,11326.000000,1.251564
124,Correction effect (pooled hybrid),BGE pooled + Node2Vec (corrected),BGE pooled + Node2Vec (uncorrected),Novelty,higher,14.035082,14.273789,1.700789
